# Multiple Linear Regression - Multi-Channel Marketing Analysis

Analyzing how TV, Radio, Social Media, and Influencer type affect Sales using a Multiple Linear Regression (OLS) model built with statsmodels.

**Why Multiple Linear Regression?** Unlike Simple Linear Regression which uses one predictor, this model lets us measure the effect of each marketing channel on Sales *while holding the others constant*. This is essential for budget allocation decisions, since marketing spend across channels is rarely independent and a manager needs to know which channel actually drives Sales after accounting for the rest.

**What this notebook covers:** data loading and cleaning, encoding categorical variables, exploratory data analysis, multicollinearity diagnostics (correlation + VIF), OLS model fitting, full statistical interpretation, assumption checking with diagnostic plots, and business recommendations tied directly to the model's coefficients.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready')

Libraries ready


## Step 1: Load and Explore the Dataset

**What we are doing:** Loading the marketing dataset and checking its shape, data types, missing values, and summary statistics.

**Why:** Before modeling, we must confirm the data is complete and understand the scale of each variable. Missing values would bias the regression, and understanding the ranges of TV, Radio, Social Media, and Sales helps us sanity-check the model coefficients later.

In [1]:
df = pd.read_csv('marketing_sales_data.csv')
print('Shape:', df.shape)
print(df.head())
print('Missing values:')
print(df.isnull().sum())
print(df.describe())

Shape: (572, 5)

       TV      Radio  Social Media Influencer       Sales
0     Low   3.518070      2.293790      Micro   55.261284
1     Low   7.756876      2.572287       Mega   67.574904
2    High  20.348988      1.227180      Micro  272.250108
3  Medium  20.108487      2.728374       Mega  195.102176
4    High  31.653200      7.776978       Nano  273.960377

Missing values:
TV              0
Radio           0
Social Media    0
Influencer      0
Sales           0

            Radio  Social Media       Sales
count  572.000000    572.000000  572.000000
mean    17.520616      3.333803  189.296908
std      9.290933      2.238378   89.871581
min      0.109106      0.000031   33.509810
25%     10.699556      1.585549  118.718722
50%     17.149517      3.150111  184.005362
75%     24.606396      4.730408  264.500118
max     42.271579     11.403625  357.788195


## Step 2: Encode Categorical Variables

TV is ordinal (Low=0, Medium=1, High=2). Influencer is nominal, one-hot encoded (dropping the first category, 'Macro', to avoid the dummy variable trap).

**Why this matters:** statsmodels OLS requires numeric input. Treating TV as ordinal (0/1/2) assumes the *step* from Low to Medium has roughly the same Sales effect as Medium to High - a reasonable assumption here since TV budget tiers are sequential. Influencer has no natural order, so one-hot encoding lets the model estimate a separate effect for each influencer tier relative to the dropped baseline (Macro).

In [1]:
df_clean = df.copy()
df_clean['TV_encoded'] = df_clean['TV'].map({'Low':0,'Medium':1,'High':2})
df_clean = pd.get_dummies(df_clean, columns=['Influencer'], drop_first=True)
for c in df_clean.select_dtypes(bool).columns:
    df_clean[c] = df_clean[c].astype(int)
print('Encoded shape:', df_clean.shape)
print(df_clean.head())

Encoded shape: (572, 8)

       TV      Radio  Social Media       Sales  TV_encoded  Influencer_Mega  Influencer_Micro  Influencer_Nano
0     Low   3.518070      2.293790   55.261284           0                0                 1                0
1     Low   7.756876      2.572287   67.574904           0                1                 0                0
2    High  20.348988      1.227180  272.250108           2                0                 1                0
3  Medium  20.108487      2.728374  195.102176           1                1                 0                0
4    High  31.653200      7.776978  273.960377           2                0                 0                1


## Step 3: Exploratory Data Analysis (EDA)

**What we are doing:** Visualizing the distribution of Sales and how it varies across TV budget level, Radio spend, Social Media spend, and Influencer type.

**Why:** EDA helps us spot patterns, outliers, and the rough shape of each relationship before fitting a model. It also gives an early, informal signal of which predictors are likely to matter, which we will later confirm or reject with formal p-values.

In [1]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].hist(df['Sales'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Sales Distribution')
df.groupby('TV')['Sales'].mean().reindex(['Low','Medium','High']).plot(kind='bar', ax=axes[1])
axes[1].set_title('Average Sales by TV Level')
plt.tight_layout()
plt.show()
print(df.groupby('TV')['Sales'].mean().round(2))

TV
High      300.85
Low       90.98
Medium    195.36
Name: Sales, dtype: float64


In [1]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(df['Radio'], df['Sales'], alpha=0.5)
axes[0].set_title('Radio vs Sales')
axes[1].scatter(df['Social Media'], df['Sales'], alpha=0.5, color='orange')
axes[1].set_title('Social Media vs Sales')
plt.tight_layout()
plt.show()
print('Radio correlation:', round(df['Radio'].corr(df['Sales']),4))
print('Social Media correlation:', round(df['Social Media'].corr(df['Sales']),4))

Radio correlation: 0.858
Social Media correlation: 0.542


In [1]:
fig, ax = plt.subplots(figsize=(8,4))
for i, inf in enumerate(['Mega','Macro','Micro','Nano']):
    data = df[df['Influencer']==inf]['Sales']
    ax.boxplot(data, positions=[i], widths=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels(['Mega','Macro','Micro','Nano'])
ax.set_title('Sales by Influencer Type')
plt.tight_layout()
plt.show()
print(df.groupby('Influencer')['Sales'].mean().round(2))

Influencer
Macro    181.67
Mega     194.49
Micro    188.32
Nano     191.87


## Step 4: Multicollinearity Check - Correlation Matrix and VIF

**What we are doing:** Checking whether our predictor variables are correlated with each other (not just with Sales).

**Why this matters:** Multiple Linear Regression assumes predictors are not strongly correlated with one another. When they are, the model struggles to separate each variable's individual effect, inflating the standard errors of the coefficients and making p-values unreliable - even if the model's overall predictive power (R-squared) stays high. We check this two ways: a correlation heatmap (pairwise) and the Variance Inflation Factor, or VIF (which captures multicollinearity from *all* other predictors combined, not just one at a time).

In [1]:
corr = df_clean[['TV_encoded','Radio','Social Media','Sales']].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print(corr)

              TV_encoded     Radio  Social Media     Sales
TV_encoded      1.000000  0.803377      0.511758  0.933169
Radio           0.803377  1.000000      0.629941  0.858036
Social Media    0.511758  0.629941      1.000000  0.542048
Sales           0.933169  0.858036      0.542048  1.000000


In [1]:
feature_cols = [c for c in ['TV_encoded','Radio','Social Media',
    'Influencer_Mega','Influencer_Micro','Influencer_Nano'] if c in df_clean.columns]
X_vif = df_clean[feature_cols].astype(float)
vif_data = pd.DataFrame({'Feature': feature_cols,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(len(feature_cols))]})
print(vif_data)
print()
print('VIF interpretation guide:')
print('  VIF < 5   : low multicollinearity (generally fine)')
print('  VIF 5-10  : moderate multicollinearity (use with caution)')
print('  VIF > 10  : high multicollinearity (problematic)')

                Feature        VIF
0           TV_encoded   6.505000
1                Radio  13.116000
2         Social Media   5.281000
3     Influencer_Mega   1.589000
4    Influencer_Micro   1.601000
5     Influencer_Nano   1.632000

VIF interpretation guide:
  VIF < 5   : low multicollinearity (generally fine)
  VIF 5-10  : moderate multicollinearity (use with caution)
  VIF > 10  : high multicollinearity (problematic)


### Interpreting Our VIF Results

Our calculated VIF values are:

- **TV_encoded = 6.505** -> moderate multicollinearity
- **Radio = 13.116** -> high multicollinearity
- **Social Media = 5.281** -> moderate multicollinearity (borderline)
- Influencer dummies (1.59 - 1.63) -> low multicollinearity, not a concern

**This means TV_encoded, Radio, and Social Media show moderate to high multicollinearity with each other**, not the 'no multicollinearity' that a blanket VIF < 5 rule would suggest. In practice this happens because higher TV budget tiers in this dataset tend to be paired with higher Radio and Social Media spend (campaigns are often planned together across channels), so the variables move together.

**Does this affect the model?** Multicollinearity does not bias the model's *overall* predictive accuracy (R-squared stays valid), but it does inflate the standard errors of the affected coefficients, making their individual p-values less trustworthy and the exact size of each coefficient less stable. This is consistent with what we see later: Radio (VIF=13.1) is still statistically significant despite high multicollinearity, but Social Media (VIF=5.3) is not significant - and its weak, unstable estimated effect is partly a symptom of this overlap with TV and Radio spend. We proceed with the model since our primary goal is identifying the strongest overall driver of Sales (TV) rather than precisely isolating the smaller effects of every channel, but this caveat is reported transparently in the final business recommendation.

## Step 5: Build the Multiple Linear Regression Model (OLS)

**What we are doing:** Fitting an Ordinary Least Squares (OLS) regression with Sales as the dependent variable and TV_encoded, Radio, Social Media, and the Influencer dummy variables as predictors.

**Why OLS:** OLS finds the coefficients that minimize the sum of squared differences between actual and predicted Sales, giving us the best linear unbiased estimate (under standard assumptions) of each predictor's effect, along with p-values to test whether each effect is statistically distinguishable from zero.

In [1]:
feature_cols = [c for c in ['TV_encoded','Radio','Social Media',
    'Influencer_Mega','Influencer_Micro','Influencer_Nano'] if c in df_clean.columns]
X = df_clean[feature_cols].astype(float)
y = df_clean['Sales']
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     887.9
Date:                Tue, 30 Jun 2026   Prob (F-statistic):          7.41e-284
Time:                        06:03:42   Log-Likelihood:                -2713.7
No. Observations:                 572   AIC:                             5441.
Df Residuals:                     565   BIC:                             5472.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               63.5316      3.332  

### Final Regression Equation

The cell below prints the fitted Multiple Linear Regression equation using the model's estimated coefficients, in standard mathematical form.

In [1]:
const = model.params['const']
terms = []
for v in feature_cols:
    c = model.params[v]
    sign = '+' if c >= 0 else '-'
    terms.append(f'{sign} {abs(c):.4f}*{v}')
equation = 'Sales = ' + f'{const:.4f} ' + ' '.join(terms)
print('Fitted Multiple Linear Regression Equation:')
print(equation)

Fitted Multiple Linear Regression Equation:
Sales = 63.5316 + 77.4451*TV_encoded + 2.9640*Radio + (-0.1469)*Social Media + 2.6192*Influencer_Mega + 2.9769*Influencer_Micro + 0.7448*Influencer_Nano


### Full Statistical Interpretation

**R-squared = 0.9041** - The model explains about 90.4% of the variation in Sales using these predictors. This is a very strong fit for a marketing dataset, meaning TV, Radio, Social Media, and Influencer type together account for most of the variability we see in Sales outcomes.

**Adjusted R-squared = 0.9031** - This is R-squared adjusted to penalize for the number of predictors in the model, so it does not automatically increase just because we added more variables. Since Adjusted R-squared (0.9031) is nearly identical to R-squared (0.9041), our predictors are genuinely contributing to the fit rather than just adding noise.

**F-statistic = 887.86, p-value = 7.41e-284** - The F-test checks whether the model as a whole is statistically meaningful (i.e., whether at least one predictor has a real effect on Sales). With a p-value this far below 0.05, we strongly reject the null hypothesis that all coefficients are zero - the model overall is highly statistically significant.

**Individual p-values (which predictors actually matter):**

- **TV_encoded**: coef = 77.4451, p < 0.0001 -> **statistically significant**. Each step up in TV budget tier (Low to Medium, or Medium to High) is associated with a Sales increase of about 77.45 units, holding all other variables constant. This is both the largest coefficient and the most significant predictor in the model.
- **Radio**: coef = 2.9640, p < 0.0001 -> **statistically significant**. Each additional unit of Radio spend is associated with a Sales increase of about 2.96 units, holding everything else constant.
- **Social Media**: coef = -0.1469, p = 0.8279 -> **not statistically significant**. The p-value is far above 0.05, so we cannot conclude Social Media spend has any real effect on Sales in this model. This lines up with its moderate-to-high VIF (5.281): some of Social Media's apparent relationship with Sales is actually being absorbed by TV and Radio, which it correlates with, leaving little independent signal.
- **Influencer_Mega, Influencer_Micro, Influencer_Nano**: p-values of 0.449, 0.378, and 0.824 respectively -> **none are statistically significant**. None of these influencer tiers show a Sales effect that is reliably different from the baseline category (Macro), once TV and Radio spend are accounted for.

**Bottom line:** Of six predictors, only **TV_encoded** and **Radio** are statistically significant drivers of Sales. Social Media and all Influencer tiers do not show a reliable independent effect once the other variables are controlled for.

## Step 6: Diagnostic Plots - Check OLS Assumptions

**What we are doing:** OLS regression relies on several assumptions about the residuals (the differences between actual and predicted Sales). If these assumptions are violated, the model's coefficients can still be calculated, but the standard errors, p-values, and confidence intervals become unreliable. We check four assumptions visually, plus one numerically (Durbin-Watson).

In [1]:
residuals = model.resid
fitted = model.fittedvalues
fig, axes = plt.subplots(2,2,figsize=(12,9))
axes[0,0].scatter(fitted, residuals, alpha=0.5)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title('Residuals vs Fitted (Linearity)')
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot (Normality)')
axes[1,0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.5)
axes[1,0].set_title('Scale-Location (Homoscedasticity)')
axes[1,1].hist(residuals, bins=25, color='orange', edgecolor='white')
axes[1,1].set_title('Residuals Histogram (Normality)')
plt.tight_layout()
plt.show()
print('Residuals mean:', round(residuals.mean(),6))
print('Residuals std:', round(residuals.std(),4))

Residuals mean: 0.0
Residuals std: 27.8298


In [1]:
from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(model.resid)
print('Durbin-Watson statistic:', round(dw_stat, 4))
print('Interpretation: values near 2 indicate no autocorrelation in residuals (independence holds).')
print('Values toward 0 suggest positive autocorrelation; values toward 4 suggest negative autocorrelation.')

Durbin-Watson statistic: 1.8762
Interpretation: values near 2 indicate no autocorrelation in residuals (independence holds).
Values toward 0 suggest positive autocorrelation; values toward 4 suggest negative autocorrelation.


### Interpreting Each Diagnostic Plot

**Residuals vs Fitted (top-left) - tests Linearity:** This plot checks whether the relationship between predictors and Sales is genuinely linear. We want the residuals scattered randomly around the zero line with no clear curve or funnel shape. Our plot shows a roughly random scatter around zero without an obvious systematic pattern, supporting the linearity assumption.

**Q-Q Plot (top-right) - tests Normality:** This compares the distribution of our residuals against a theoretical normal distribution. If residuals are normally distributed, the points fall closely along the diagonal reference line. Points that deviate strongly in the tails would indicate non-normal residuals (often caused by outliers), which can affect the reliability of p-values and confidence intervals. Our residuals largely follow the diagonal line in the central range, with only minor deviation in the extreme tails - acceptable for a dataset of this size.

**Scale-Location (bottom-left) - tests Homoscedasticity:** This plot checks whether residual variance is constant across all fitted values (homoscedasticity), as opposed to increasing or decreasing (heteroscedasticity). We want a roughly flat, even spread of points across the x-axis. A funnel or fan shape would signal heteroscedasticity, meaning our model is less accurate at certain Sales levels than others. Our plot shows a reasonably even spread without a strong funnel pattern.

**Residual Histogram (bottom-right) - tests Normality:** This gives a direct view of the shape of the residual distribution. We want an approximately bell-shaped, symmetric distribution centered at zero. Our histogram is roughly bell-shaped and centered near zero, consistent with the Q-Q plot's conclusion of approximate normality.

**Durbin-Watson statistic - tests Independence of residuals:** This statistic ranges from 0 to 4, where a value near 2 indicates no autocorrelation (residuals are independent of each other), values toward 0 suggest positive autocorrelation, and values toward 4 suggest negative autocorrelation. Our Durbin-Watson value is **1.876**, which is very close to 2, indicating the residuals are not meaningfully autocorrelated and the independence assumption holds reasonably well.

**Overall assumption check conclusion:** Linearity, normality, and homoscedasticity all hold reasonably well based on the diagnostic plots, and independence is supported by the Durbin-Watson statistic. The main caveat to this model is not an OLS assumption violation, but the moderate-to-high multicollinearity identified earlier in the VIF analysis, which affects how confidently we can interpret the Social Media and Influencer coefficients specifically.

## Step 7: Interpret Coefficients and Give Business Recommendations

**What we are doing:** Translating the statistically significant coefficients into concrete, prioritized marketing budget guidance, while being explicit about which recommendations are backed by strong statistical evidence and which are not.

In [1]:
print('R-squared:', round(model.rsquared,4), '| Adjusted R-squared:', round(model.rsquared_adj,4))
print('F-statistic:', round(model.fvalue,2), '| F p-value:', model.f_pvalue)
print()
print('Coefficient summary:')
for v in model.params.index:
    c = round(float(model.params[v]),4)
    p = round(float(model.pvalues[v]),6)
    sig = 'Significant' if p < 0.05 else 'Not significant'
    print(v, 'coef='+str(c), 'pval='+str(p), sig)
print()
print('BUSINESS RECOMMENDATIONS (tied to coefficients and significance):')
print('1. Prioritize TV budget. coef=77.45, p<0.0001 (significant). Each TV tier increase')
print('   (Low to Medium to High) is linked to a ~77.45 unit Sales increase, holding other')
print('   channels constant. This is the single strongest, most reliable lever available.')
print('2. Maintain or grow Radio spend. coef=2.96, p<0.0001 (significant). Each additional')
print('   unit of Radio spend adds ~2.96 units of Sales. Smaller than TV per unit, but still')
print('   a statistically reliable, positive contributor.')
print('3. Do not increase Social Media spend based on this model alone. coef=-0.15,')
print('   p=0.828 (not significant). The model finds no reliable Sales effect from Social')
print('   Media once TV and Radio are accounted for; treat any apparent benefit with caution')
print('   given its VIF of 5.28 indicates overlap with the other channels.')
print('4. Influencer tier choice should not be driven by this model. None of Mega, Micro,')
print('   or Nano showed a significant Sales difference versus the baseline (Macro),')
print('   p-values all above 0.37. Influencer selection should be based on other factors')
print('   (cost, brand fit) rather than an expected Sales lift.')
print('5. Caveat: Radio (VIF=13.1) and TV_encoded (VIF=6.5) show meaningful multicollinearity,')
print('   so the exact size of each coefficient should be treated as indicative rather than')
print('   precise. The overall conclusion that TV is the dominant driver remains robust,')
print('   since it stayed significant and large despite this overlap.')

R-squared: 0.9041 | Adjusted R-squared: 0.9031
F-statistic: 887.86 | F p-value: 7.406237034492272e-284

Coefficient summary:
const coef=63.5316 pval=0.0 Significant
TV_encoded coef=77.4451 pval=0.0 Significant
Radio coef=2.964 pval=0.0 Significant
Social Media coef=-0.1469 pval=0.82791 Not significant
Influencer_Mega coef=2.6192 pval=0.448793 Not significant
Influencer_Micro coef=2.9769 pval=0.37822 Not significant
Influencer_Nano coef=0.7448 pval=0.82377 Not significant

BUSINESS RECOMMENDATIONS (tied to coefficients and significance):
1. Prioritize TV budget. coef=77.45, p<0.0001 (significant). Each TV tier increase
   (Low to Medium to High) is linked to a ~77.45 unit Sales increase, holding other
   channels constant. This is the single strongest, most reliable lever available.
2. Maintain or grow Radio spend. coef=2.96, p<0.0001 (significant). Each additional
   unit of Radio spend adds ~2.96 units of Sales. Smaller than TV per unit, but still
   a statistically reliable, positive